# Tâche 2 : Question-réponse avec poursuite du pré-entraînement du modèle génératif

Cette tâche consiste à poursuivre le préentraînement d'un modèle pré‑entraîné sur un corpus de petite taille. On vous demande d'évaluer la qualité des réponses du modèle après avoir complété l'entraînement supplémentaire. L'évaluation du modèle est effectué avec les mêmes paires **questions-réponses** que la tâche 1. 

NOTE: Il est important de sauvegarder le modèle pré-entraîné dans cette tâche (ainsi que le tokenizer) car vous les réutilisez à la tâche 3. 

> Il est recommandé de faire ce travail pratique en utilisant une carte graphique GPU compatible avec HuggingFace/Pytorch.
> Si votre machine n’en possède pas, vous pouvez utiliser **Google Colab** pour exécuter le *notebook* dans le cloud.

In [1]:
#!pip install transformers
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    pipeline,
    DataCollatorForLanguageModeling,
)
from transformers.pipelines.pt_utils import KeyDataset
from datasets import load_dataset
import torch
import json
import os
import time

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement du modèle Hugging Face et du tokenizer

In [3]:
model_name = "EleutherAI/pythia-410m"

tokenizer = AutoTokenizer.from_pretrained(model_name,cache_dir="saved_models/pythia-410m")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.bfloat16, cache_dir="saved_models/pythia-410m")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 292/292 [00:00<00:00, 1097.10it/s]


In [4]:
# test
inputs = tokenizer("Hello, I am", return_tensors="pt", max_length=24)
tokens = model.generate(**inputs)
print(tokenizer.decode(tokens[0]))

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Hello, I am a newbie to this site and I am trying to get a list of all the products that are


## 2. Fonctions de test question-réponse

In [5]:
def build_prompt(question, context=None):
    few_shot = (
        "Question: What mechanism allows a Transformer model to weigh the importance of different words in a sequence?\n"
        "Answer: The self-attention mechanism allows a Transformer to weigh the importance of different tokens relative to each other.\n\n"
        "Question: What is the purpose of positional encodings in the Transformer architecture?\n"
        "Answer: Positional encodings inject information about the relative or absolute position of tokens in the sequence.\n\n"
    )
    if context:
        return f"Answer concisely and factually.\n\n{few_shot}Context: {context}\nQuestion: {question}\nAnswer:"
    return f"Answer concisely and factually.\n\n{few_shot}Question: {question}\nAnswer:"

def clean_generation(generated_text):
    text = generated_text.strip()
    for stop_token in ["\nQuestion:", "\nQ:", "\n###", "\n\n"]:
        if stop_token in text:
            text = text.split(stop_token)[0].strip()
    return text

## 3. Poursuite du pré-entraînement du modèle

In [6]:
block_size = 256

dataset = load_dataset("json", data_files="data/train_texts.json")

def tokenize_function(examples):
    texts = [f"{title}\n{text}" for title, text in zip(examples["title"], examples["text"])]
    #texts = [f"{text}" for title, text in zip(examples["title"], examples["text"])]
    ids = tokenizer(texts)["input_ids"]
    return {"input_ids": [x + [tokenizer.eos_token_id] for x in ids]}

def group_texts(examples):
    concat = sum(examples["input_ids"], [])
    total = (len(concat) // block_size) * block_size
    chunks = [concat[i:i+block_size] for i in range(0, total, block_size)]
    return {"input_ids": chunks, "labels": [c.copy() for c in chunks]}

tok = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)
lm_dataset = tok.map(group_texts, batched=True)
split = lm_dataset["train"].train_test_split(test_size=0.05)

print(f"Nombre d'exemples pour l'entraînement : {len(split['train'])}")

Map: 100%|██████████| 747/747 [00:00<00:00, 6245.67 examples/s]

Nombre d'exemples pour l'entraînement : 384


In [7]:
tokenizer.padding_side = "right"

training_args = TrainingArguments(
    output_dir="checkpoints/pythia-410m-finetuned-t2",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    warmup_steps=0.1,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,
    max_grad_norm=1.0,
    bf16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    processing_class=tokenizer
)

print(trainer.evaluate())

Training Loss,Validation Loss,Epoch
No log,3.602967,0


{'eval_loss': 3.602966785430908}


In [8]:
inputs = tokenizer("GPT is a ", return_tensors="pt", max_new_tokens=50, max_length=50)
tokens = model.generate(**inputs)
print(tokenizer.decode(tokens[0]))

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


GPT is a 
not-so-simple task.

~~~
jamesbritt
I think the


In [9]:
trainer.train()

trainer.save_model("saved_models/pythia-410m-final-t2")
tokenizer.save_pretrained("saved_models/pythia-410m-final-t2")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss
1,3.115100,3.186988
2,3.013517,3.125489
3,2.823000,3.116190
4,2.877488,3.113555


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
[transformers] There were unexpected keys in the checkpoint model loaded: ['embed_out.weight'].
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.75it/s]


('saved_models/pythia-410m-final-t2\\tokenizer_config.json',
 'saved_models/pythia-410m-final-t2\\tokenizer.json')

In [10]:
inputs = tokenizer("GPT is a ", return_tensors="pt", max_new_tokens=50, max_length=50)
tokens = model.generate(**inputs, repetition_penalty=1.2)
print(tokenizer.decode(tokens[0]))

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


GPT is a  model that has been trained on the CIFAR-10 dataset. It was released in June


## 4. Génération de réponses avec le nouveau modèle

In [11]:
#for test
#model = AutoModelForCausalLM.from_pretrained("saved_models/pythia-410m-final-t2")
#tokenizer = AutoTokenizer.from_pretrained("saved_models/pythia-410m-final-t2")

In [12]:
test_fn = "data/test_questions.json"

raw_dataset = load_dataset("json", data_files=test_fn)["train"]

dataset = raw_dataset.map(
    lambda example: {"prompt": build_prompt(example["question"])},
)
print(f"Nombre de questions chargées dans le Dataset : {len(dataset)}")

Nombre de questions chargées dans le Dataset : 50


In [13]:
pipe_dataset = KeyDataset(dataset, "prompt")

raw_outputs = []
start_time = time.time()

if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

for out in generator(
    pipe_dataset,
    batch_size=16,
    max_new_tokens=264,
    do_sample=True,
    repetition_penalty=1.05,
    top_p=0.7,
    temperature=0.5,
    stop_strings=["\n", "Question:"],
    tokenizer=tokenizer,
    return_full_text=False
):
    raw_outputs.append(out[0]["generated_text"])

print(f"Génération en batch complétée en {time.time() - start_time:.2f} secondes !")

results = []
for sample, raw_gen in zip(dataset, raw_outputs):
    results.append({
        "Q": sample["question"],
        "A": clean_generation(raw_gen),
        "Expected": sample.get("answer", "")
    })

out_dir = "result"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "t2_results.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Fichier de résultats sauvegardé ({len(results)} entrées) : {out_path}")

Génération en batch complétée en 7.36 secondes !
Fichier de résultats sauvegardé (50 entrées) : result\t2_results.json


## 5. Évaluation des résultats 

Vous pouvez réutiliser le code d'évaluation rendu disponible pour la tâche 1.

In [14]:
import re
from collections import Counter
import evaluate
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def tokenize_no_stopwords(text: str):
    tokens = re.findall(r"\b\w+\b", text.lower())
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS]


def compute_word_metrics(results: list[dict]):
    precisions, recalls, f1s = [], [], []

    for item in results:
        pred_tokens = tokenize_no_stopwords(item["A"])
        ref_tokens = tokenize_no_stopwords(item["Expected"])

        common = Counter(pred_tokens) & Counter(ref_tokens)
        num_same = sum(common.values())

        if len(pred_tokens) == 0 or len(ref_tokens) == 0 or num_same == 0:
            precisions.append(0.0)
            recalls.append(0.0)
            f1s.append(0.0)
            continue

        p = num_same / len(pred_tokens)
        r = num_same / len(ref_tokens)
        f1 = (2 * p * r) / (p + r)

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    n = len(results)
    avg_p = (sum(precisions) / n) * 100 if n > 0 else 0.0
    avg_r = (sum(recalls) / n) * 100 if n > 0 else 0.0
    avg_f1 = (sum(f1s) / n) * 100 if n > 0 else 0.0
    return avg_p, avg_r, avg_f1


squad = evaluate.load("squad")

squad_predictions = [{"id": str(i), "prediction_text": item["A"]} for i, item in enumerate(results)]
squad_references = [{"id": str(i), "answers": {"text": [item["Expected"]], "answer_start": [0]}} for i, item in enumerate(results)]

metrics = squad.compute(predictions=squad_predictions, references=squad_references)
word_precision, word_recall, word_f1 = compute_word_metrics(results)

print("=" * 60)
print(" RÉSULTATS EVALUATE:")
print(f" Exact Match (SQuAD) : {metrics['exact_match']:.2f}%")
print(f" F1-Score (SQuAD)    : {metrics['f1']:.2f}%")
print("-" * 60)
print(f" Précision (Mots)    : {word_precision:.2f}%")
print(f" Rappel (Mots)       : {word_recall:.2f}%")
print(f" F1-Score (Mots)     : {word_f1:.2f}%")
print("=" * 60)

 RÉSULTATS EVALUATE:
 Exact Match (SQuAD) : 0.00%
 F1-Score (SQuAD)    : 23.58%
------------------------------------------------------------
 Précision (Mots)    : 25.08%
 Rappel (Mots)       : 23.24%
 F1-Score (Mots)     : 21.92%


In [15]:
print("Exemples de réponses générées :")
for sample in results[:3]:
    print("-" * 50)
    print(f"Q        : {sample['Q']}")
    print(f"Expected : {sample['Expected']}")
    print(f"A (gen)  : {sample['A']}")

Exemples de réponses générées :
--------------------------------------------------
Q        : What is the context window size of GPT-3?
Expected : GPT-3 has a context window size of 2,048 tokens.
A (gen)  : The context window size is the number of tokens that are fed into the transformer before it starts training.
--------------------------------------------------
Q        : Explain what is a transformer architecture.
Expected : The transformer architecture is a neural network based on a deep learning model that was introduced in 2017, used in natural language processing (NLP) systems.
A (gen)  : A transformer architecture is a neural network that transforms an input sequence into another sequence. It can be used for language modeling, text generation, or any other task.
--------------------------------------------------
Q        : Define what is a generative pre-trained transformer (GPT).
Expected : A generative pre-trained transformer is a type of generative large language model that